# Advanced EDA and Quality Control for TP53 Mutation Prediction

This notebook adds focused exploratory analyses that are useful before prediction:

1. Data audit and label sanity checks.
2. Sample-level and gene-level sparsity.
3. PCA scree plots and PCA colored by TP53 status, mutation type, and lineage.
4. Multi-method outlier detection.
5. Lineage-aware mutation summaries.
6. Canonical p53 target expression checks.
7. Exploratory sample correlation and clustering.

It is intentionally separate from `main_notebook.ipynb` so you can review the figures and decide which pieces should be moved into the final report.

## How to Run

Run cells from top to bottom.

The notebook first tries to load processed files from `data/processed/`. If they are missing, it tries to build them from raw DepMap files in `data/raw/`. If both are missing, run `python scripts/run_pipeline.py` first, or set `AUTO_DOWNLOAD = True` in the data-loading cell if you want this notebook to download the configured DepMap files.

Figures are saved in `reports/figures/advanced_eda/`; tables are saved in `reports/tables/advanced_eda/`.

## 0. Setup

In [ ]:
from pathlib import Path
import os
import sys
import warnings

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from scipy.stats import mannwhitneyu
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import adjusted_rand_score, confusion_matrix, silhouette_score
from sklearn.preprocessing import StandardScaler

from tp53_ml.config import load_config
from tp53_ml.data import (
    align_metadata,
    download_depmap_file,
    make_tp53_labels,
    read_expression,
    read_metadata,
    try_download_candidates,
)
from tp53_ml.genes import CANONICAL_P53_TARGETS_INFO, P53_PATHWAY_GENES
from tp53_ml.preprocessing import TopVarianceSelector

sns.set_theme(style="whitegrid", context="notebook")

cfg = load_config("config/project.yaml")
RANDOM_STATE = cfg["project"]["random_state"]

FIG_DIR = Path("reports/figures/advanced_eda")
TABLE_DIR = Path("reports/tables/advanced_eda")
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Random state:", RANDOM_STATE)
print("Figure output:", FIG_DIR)
print("Table output:", TABLE_DIR)

## 1. Load Data

In [ ]:
AUTO_DOWNLOAD = False

raw_dir = Path(cfg["data"]["raw_dir"])
expr_processed = Path(cfg["data"]["expression_processed"])
labels_processed = Path(cfg["data"]["labels_processed"])
metadata_processed = Path(cfg["data"].get("metadata_processed", "data/processed/sample_metadata.csv"))


def find_existing(raw_dir, candidates):
    for candidate in candidates:
        path = raw_dir / candidate
        if path.exists() and path.stat().st_size > 1024:
            return path
    return None


if expr_processed.exists() and labels_processed.exists():
    X = pd.read_csv(expr_processed, index_col=0)
    labels = pd.read_csv(labels_processed)
    metadata = pd.read_csv(metadata_processed) if metadata_processed.exists() else pd.DataFrame({"sample_id": X.index.astype(str)})
    data_source = "processed"
else:
    release = cfg["depmap"]["release"]
    expression_path = raw_dir / cfg["depmap"]["expression_file"]
    mutation_path = find_existing(raw_dir, cfg["depmap"]["mutation_file_candidates"])
    metadata_path = find_existing(raw_dir, cfg["depmap"]["metadata_file_candidates"])

    if AUTO_DOWNLOAD:
        raw_dir.mkdir(parents=True, exist_ok=True)
        expression_path = download_depmap_file(release, cfg["depmap"]["expression_file"], raw_dir)
        mutation_path = try_download_candidates(release, cfg["depmap"]["mutation_file_candidates"], raw_dir)
        metadata_path = try_download_candidates(release, cfg["depmap"]["metadata_file_candidates"], raw_dir)

    missing = []
    if not expression_path.exists():
        missing.append(str(expression_path))
    if mutation_path is None:
        missing.append("one of " + str(cfg["depmap"]["mutation_file_candidates"]))
    if metadata_path is None:
        missing.append("one of " + str(cfg["depmap"]["metadata_file_candidates"]))
    if missing:
        raise FileNotFoundError(
            "No usable local data files found. Run `python scripts/run_pipeline.py` first, "
            "or set AUTO_DOWNLOAD = True in this cell. Missing: " + "; ".join(missing)
        )

    X = read_expression(expression_path)
    mutations = pd.read_csv(mutation_path, low_memory=False)
    labels = make_tp53_labels(mutations, X.index)
    metadata = align_metadata(read_metadata(metadata_path), X.index)
    data_source = "raw"

X.index = X.index.astype(str)
labels = labels.copy()
labels["sample_id"] = labels["sample_id"].astype(str)
metadata = metadata.copy()
if "sample_id" not in metadata.columns:
    metadata["sample_id"] = X.index.astype(str)
metadata["sample_id"] = metadata["sample_id"].astype(str)

label_df = labels.set_index("sample_id").reindex(X.index)
if label_df["tp53_mutant"].isna().any():
    raise ValueError("Some expression samples do not have TP53 labels after alignment.")

meta_df = metadata.drop_duplicates("sample_id").set_index("sample_id").reindex(X.index)
lineage_col = next(
    (c for c in ["OncotreeLineage", "onCotreeLineage", "lineage", "primary_disease", "CCLE_Name"] if c in meta_df.columns),
    None,
)

analysis_df = label_df.copy()
analysis_df["tp53_status"] = analysis_df["tp53_mutant"].map({0: "WT", 1: "Mutant"})
analysis_df["lineage"] = meta_df[lineage_col].fillna("Unknown").astype(str) if lineage_col else "Unknown"
analysis_df["mutation_type_collapsed"] = analysis_df["mutation_type_collapsed"].astype(str)

print(f"Loaded from: {data_source}")
print(f"Expression matrix: {X.shape[0]} samples x {X.shape[1]} genes")
print(f"Lineage column: {lineage_col if lineage_col else 'not available'}")
display(analysis_df.head())

## 2. Data Audit

This section checks the main dimensions, label balance, and metadata coverage before any exploratory plot. These numbers are useful in the report because they make the dataset construction transparent.

In [ ]:
audit_rows = [
    {"metric": "samples", "value": X.shape[0]},
    {"metric": "genes", "value": X.shape[1]},
    {"metric": "missing_expression_values", "value": int(X.isna().sum().sum())},
    {"metric": "duplicated_gene_names", "value": int(pd.Index(X.columns).duplicated().sum())},
    {"metric": "mutant_samples", "value": int((analysis_df["tp53_mutant"] == 1).sum())},
    {"metric": "wt_samples", "value": int((analysis_df["tp53_mutant"] == 0).sum())},
    {"metric": "lineages", "value": int(analysis_df["lineage"].nunique())},
]
audit = pd.DataFrame(audit_rows)
audit.to_csv(TABLE_DIR / "data_audit.csv", index=False)

display(audit)

display(
    analysis_df["tp53_status"]
    .value_counts()
    .rename_axis("tp53_status")
    .reset_index(name="samples")
)

display(
    analysis_df["mutation_type_collapsed"]
    .value_counts()
    .rename_axis("mutation_type")
    .reset_index(name="samples")
)

## 3. Sample-Level Sparsity

Sparsity is the fraction of zero or near-zero expression values. With log2(TPM + 1), exact zeros usually mean no observed expression. If TP53-mutant and WT samples differ in global sparsity, that is a useful QC and biology signal.

In [ ]:
NEAR_ZERO_THRESHOLD = 0.0

zero_mask = X <= NEAR_ZERO_THRESHOLD
sample_sparsity = zero_mask.mean(axis=1)

sample_sparsity_df = analysis_df[["tp53_status", "tp53_mutant", "mutation_type_collapsed", "lineage"]].copy()
sample_sparsity_df["sample_id"] = sample_sparsity_df.index
sample_sparsity_df["sample_sparsity"] = sample_sparsity.values
sample_sparsity_df["sample_mean"] = X.mean(axis=1).values
sample_sparsity_df["sample_std"] = X.std(axis=1).values
sample_sparsity_df.to_csv(TABLE_DIR / "sample_sparsity.csv", index=False)

mut_s = sample_sparsity_df.loc[sample_sparsity_df["tp53_mutant"] == 1, "sample_sparsity"]
wt_s = sample_sparsity_df.loc[sample_sparsity_df["tp53_mutant"] == 0, "sample_sparsity"]
if len(mut_s) > 0 and len(wt_s) > 0:
    u_stat, p_val = mannwhitneyu(mut_s, wt_s, alternative="two-sided")
else:
    u_stat, p_val = np.nan, np.nan

summary = sample_sparsity_df.groupby("tp53_status")["sample_sparsity"].agg(["count", "mean", "median", "std"])
summary["mannwhitney_u"] = u_stat
summary["mannwhitney_p"] = p_val
summary.to_csv(TABLE_DIR / "sample_sparsity_by_status.csv")
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=sample_sparsity_df, x="tp53_status", y="sample_sparsity", hue="tp53_status", legend=False, ax=axes[0])
axes[0].set_title("Sample-level sparsity by TP53 status")
axes[0].set_xlabel("")
axes[0].set_ylabel("Fraction of zero expression values")

sns.histplot(data=sample_sparsity_df, x="sample_sparsity", hue="tp53_status", kde=True, element="step", ax=axes[1])
axes[1].set_title("Sample-level sparsity distribution")
axes[1].set_xlabel("Fraction of zero expression values")
fig.tight_layout()
fig.savefig(FIG_DIR / "sample_sparsity_by_tp53_status.png", dpi=180)
plt.show()

## 4. Gene-Level Sparsity and Sparsity Shifts

Gene-level sparsity asks how often each gene is zero across samples. A large difference between mutant and WT samples can highlight genes that are turned on or off more often in one group.

In [ ]:
gene_sparsity_all = zero_mask.mean(axis=0)
gene_sparsity_mut = zero_mask.loc[analysis_df["tp53_mutant"] == 1].mean(axis=0)
gene_sparsity_wt = zero_mask.loc[analysis_df["tp53_mutant"] == 0].mean(axis=0)

gene_sparsity_df = pd.DataFrame({
    "gene": X.columns,
    "sparsity_all": gene_sparsity_all.values,
    "sparsity_mutant": gene_sparsity_mut.values,
    "sparsity_wt": gene_sparsity_wt.values,
})
gene_sparsity_df["sparsity_shift_wt_minus_mutant"] = gene_sparsity_df["sparsity_wt"] - gene_sparsity_df["sparsity_mutant"]
gene_sparsity_df["abs_shift"] = gene_sparsity_df["sparsity_shift_wt_minus_mutant"].abs()
gene_sparsity_df = gene_sparsity_df.sort_values("abs_shift", ascending=False)
gene_sparsity_df.to_csv(TABLE_DIR / "gene_sparsity_shifts.csv", index=False)

display(gene_sparsity_df.head(20))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_df = pd.concat([
    pd.DataFrame({"sparsity": gene_sparsity_mut.values, "group": "Mutant"}),
    pd.DataFrame({"sparsity": gene_sparsity_wt.values, "group": "WT"}),
], ignore_index=True)
sns.histplot(data=plot_df, x="sparsity", hue="group", element="step", stat="density", common_norm=False, ax=axes[0])
axes[0].set_title("Gene-level sparsity distribution")
axes[0].set_xlabel("Fraction of samples with zero expression")

top_shift = gene_sparsity_df.head(20).iloc[::-1]
colors = np.where(top_shift["sparsity_shift_wt_minus_mutant"] >= 0, "#4C78A8", "#E45756")
axes[1].barh(top_shift["gene"], top_shift["sparsity_shift_wt_minus_mutant"], color=colors)
axes[1].axvline(0, color="black", linewidth=1)
axes[1].set_title("Largest sparsity shifts")
axes[1].set_xlabel("WT sparsity minus mutant sparsity")
fig.tight_layout()
fig.savefig(FIG_DIR / "gene_sparsity_shifts.png", dpi=180)
plt.show()

## 5. PCA Feature Space

PCA summarizes high-dimensional expression variation. We use top variable genes to keep the plot interpretable and computationally stable. PCA is unsupervised, so this is exploratory rather than model evidence.

In [ ]:
selector = TopVarianceSelector(
    k=min(cfg["features"]["top_variable_genes"], X.shape[1]),
    min_mean_expression=cfg["features"]["min_mean_expression"],
)
X_var = selector.fit_transform(X)
selected_genes = list(selector.get_feature_names_out())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_var)

n_pcs = min(50, X_scaled.shape[0] - 1, X_scaled.shape[1])
pca = PCA(n_components=n_pcs, random_state=RANDOM_STATE)
pcs = pca.fit_transform(X_scaled)

pca_df = analysis_df[["tp53_status", "tp53_mutant", "mutation_type_collapsed", "lineage"]].copy()
pca_df["sample_id"] = pca_df.index
for i in range(min(5, n_pcs)):
    pca_df[f"PC{i+1}"] = pcs[:, i]

pca_variance = pd.DataFrame({
    "component": np.arange(1, n_pcs + 1),
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "cumulative_variance_ratio": np.cumsum(pca.explained_variance_ratio_),
})
pca_variance.to_csv(TABLE_DIR / "pca_explained_variance.csv", index=False)
pca_df.to_csv(TABLE_DIR / "pca_coordinates.csv", index=False)

display(pca_variance.head(12))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(pca_variance["component"].head(20), pca_variance["explained_variance_ratio"].head(20), color="#4C78A8")
ax.plot(pca_variance["component"].head(20), pca_variance["cumulative_variance_ratio"].head(20), color="#E45756", marker="o")
ax.set_title("PCA explained variance")
ax.set_xlabel("Principal component")
ax.set_ylabel("Variance ratio")
fig.tight_layout()
fig.savefig(FIG_DIR / "pca_scree.png", dpi=180)
plt.show()

In [ ]:
plot_df = pca_df.copy()
top_lineages = plot_df["lineage"].value_counts().head(10).index
plot_df["lineage_plot"] = np.where(plot_df["lineage"].isin(top_lineages), plot_df["lineage"], "Other")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.scatterplot(data=plot_df, x="PC1", y="PC2", hue="tp53_status", alpha=0.75, s=35, ax=axes[0])
axes[0].set_title("PCA by TP53 status")

sns.scatterplot(data=plot_df, x="PC1", y="PC2", hue="mutation_type_collapsed", alpha=0.75, s=35, ax=axes[1])
axes[1].set_title("PCA by mutation type")
axes[1].legend(loc="best", fontsize=8)

sns.scatterplot(data=plot_df, x="PC1", y="PC2", hue="lineage_plot", alpha=0.75, s=35, ax=axes[2])
axes[2].set_title("PCA by lineage")
axes[2].legend(loc="best", fontsize=7)

for ax in axes:
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}% var.)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}% var.)")

fig.tight_layout()
fig.savefig(FIG_DIR / "pca_by_status_type_lineage.png", dpi=180)
plt.show()

## 6. Multi-Method Outlier Detection

Outliers are flagged three ways:

1. Isolation Forest on PCA coordinates.
2. Isolation Forest on row-level summary statistics.
3. Mahalanobis distance on PCA coordinates.

The combined flag requires at least two methods to agree. Treat this as a diagnostic: do not remove samples from final modeling unless you also report the effect on class balance and performance.

In [ ]:
row_stats = pd.DataFrame(index=X.index)
row_stats["mean"] = X.mean(axis=1)
row_stats["std"] = X.std(axis=1)
row_stats["median"] = X.median(axis=1)
row_stats["mad"] = X.sub(X.median(axis=1), axis=0).abs().median(axis=1)
row_stats["skew"] = X.skew(axis=1)
row_stats["kurtosis"] = X.kurtosis(axis=1)
row_stats["zero_pct"] = zero_mask.mean(axis=1)
row_stats = row_stats.replace([np.inf, -np.inf], np.nan).fillna(0)

contamination = min(0.05, max(1 / len(X), 0.01))

iso_pca = IsolationForest(n_estimators=200, contamination=contamination, random_state=RANDOM_STATE, n_jobs=-1)
outlier_pca = iso_pca.fit_predict(pcs[:, :min(20, n_pcs)]) == -1

iso_stats = IsolationForest(n_estimators=200, contamination=contamination, random_state=RANDOM_STATE, n_jobs=-1)
outlier_stats = iso_stats.fit_predict(row_stats) == -1

pcs_for_mahal = pcs[:, :min(20, n_pcs)]
centered = pcs_for_mahal - pcs_for_mahal.mean(axis=0)
inv_cov = np.linalg.pinv(np.cov(pcs_for_mahal, rowvar=False))
mahal_dist = np.sqrt(np.sum(centered @ inv_cov * centered, axis=1))
mahal_threshold = mahal_dist.mean() + 3 * mahal_dist.std()
outlier_mahal = mahal_dist > mahal_threshold

outlier_summary = analysis_df[["tp53_status", "mutation_type_collapsed", "lineage"]].copy()
outlier_summary["sample_id"] = outlier_summary.index
outlier_summary["outlier_isolation_pca"] = outlier_pca
outlier_summary["outlier_isolation_stats"] = outlier_stats
outlier_summary["outlier_mahalanobis"] = outlier_mahal
outlier_summary["mahalanobis_distance"] = mahal_dist
outlier_summary["outlier_vote_count"] = (
    outlier_summary[["outlier_isolation_pca", "outlier_isolation_stats", "outlier_mahalanobis"]]
    .astype(int)
    .sum(axis=1)
)
outlier_summary["combined_outlier"] = outlier_summary["outlier_vote_count"] >= 2
outlier_summary.to_csv(TABLE_DIR / "outlier_summary.csv", index=False)

counts = outlier_summary[["outlier_isolation_pca", "outlier_isolation_stats", "outlier_mahalanobis", "combined_outlier"]].sum().rename("flagged_samples")
display(counts.to_frame())

display(
    outlier_summary.loc[outlier_summary["combined_outlier"]]
    .sort_values(["outlier_vote_count", "mahalanobis_distance"], ascending=False)
    .head(30)
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_df = pca_df.copy()
plot_df["combined_outlier"] = outlier_summary.set_index("sample_id").loc[plot_df["sample_id"], "combined_outlier"].values
sns.scatterplot(data=plot_df, x="PC1", y="PC2", hue="combined_outlier", style="tp53_status", s=45, alpha=0.8, ax=axes[0])
axes[0].set_title("Combined outliers in PCA space")

sns.histplot(outlier_summary["mahalanobis_distance"], bins=40, ax=axes[1], color="#4C78A8")
axes[1].axvline(mahal_threshold, color="#E45756", linestyle="--", label="mean + 3 sd")
axes[1].set_title("Mahalanobis distance distribution")
axes[1].set_xlabel("Distance")
axes[1].legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "outlier_detection.png", dpi=180)
plt.show()

stats_to_plot = ["mean", "std", "skew", "kurtosis", "zero_pct"]
fig, axes = plt.subplots(1, len(stats_to_plot), figsize=(4 * len(stats_to_plot), 3.5))
for ax, stat in zip(axes, stats_to_plot):
    sns.boxplot(data=row_stats.assign(combined_outlier=outlier_summary["combined_outlier"].values), x="combined_outlier", y=stat, ax=ax)
    ax.set_title(stat)
    ax.set_xlabel("Outlier")
fig.tight_layout()
fig.savefig(FIG_DIR / "outlier_row_statistics.png", dpi=180)
plt.show()

## 7. Lineage-Aware EDA

Cancer lineage can drive global expression and TP53 mutation frequency. These plots show whether some lineages dominate the mutant or WT group, which is essential context for interpreting predictive performance.

In [ ]:
if analysis_df["lineage"].nunique() <= 1 or set(analysis_df["lineage"].unique()) == {"Unknown"}:
    print("No usable lineage metadata available. Skipping lineage-aware plots.")
else:
    lineage_summary = (
        analysis_df.groupby("lineage")
        .agg(
            samples=("tp53_mutant", "size"),
            mutant_samples=("tp53_mutant", "sum"),
            mutant_fraction=("tp53_mutant", "mean"),
        )
        .sort_values("samples", ascending=False)
    )
    lineage_summary.to_csv(TABLE_DIR / "lineage_tp53_summary.csv")
    display(lineage_summary.head(20))

    top_lineages = lineage_summary.head(15).index
    plot_lineage = analysis_df.loc[analysis_df["lineage"].isin(top_lineages)].copy()

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    order_by_count = lineage_summary.loc[top_lineages].index
    sns.barplot(
        data=lineage_summary.loc[top_lineages].reset_index(),
        y="lineage",
        x="samples",
        color="#4C78A8",
        order=order_by_count,
        ax=axes[0],
    )
    axes[0].set_title("Sample count by lineage")
    axes[0].set_xlabel("Samples")
    axes[0].set_ylabel("")

    order_by_frac = lineage_summary.loc[top_lineages].sort_values("mutant_fraction", ascending=False).index
    sns.barplot(
        data=lineage_summary.loc[top_lineages].reset_index(),
        y="lineage",
        x="mutant_fraction",
        color="#E45756",
        order=order_by_frac,
        ax=axes[1],
    )
    axes[1].set_title("TP53 mutant fraction by lineage")
    axes[1].set_xlabel("Mutant fraction")
    axes[1].set_ylabel("")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "lineage_counts_and_mutant_fraction.png", dpi=180)
    plt.show()

    type_by_lineage = (
        plot_lineage.groupby(["lineage", "mutation_type_collapsed"])
        .size()
        .unstack(fill_value=0)
    )
    type_by_lineage = type_by_lineage.loc[order_by_count]
    type_by_lineage_frac = type_by_lineage.div(type_by_lineage.sum(axis=1), axis=0)
    type_by_lineage_frac.to_csv(TABLE_DIR / "lineage_mutation_type_fraction.csv")

    ax = type_by_lineage_frac.plot(kind="barh", stacked=True, figsize=(10, 7), colormap="tab20")
    ax.set_title("Mutation-type composition by lineage")
    ax.set_xlabel("Fraction within lineage")
    ax.set_ylabel("")
    ax.legend(title="Mutation type", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "lineage_mutation_type_composition.png", dpi=180)
    plt.show()

## 8. Canonical p53 Target Expression

These genes are biologically interpretable TP53 targets. The goal is to see whether canonical targets already show visible expression differences between TP53-mutant and WT samples before modeling.

In [ ]:
canonical_genes = [gene for gene in CANONICAL_P53_TARGETS_INFO if gene in X.columns]
print(f"Canonical p53 targets present: {len(canonical_genes)} / {len(CANONICAL_P53_TARGETS_INFO)}")
print(canonical_genes)

if not canonical_genes:
    raise ValueError("None of the canonical p53 target genes were found in X.columns.")

canonical_long = (
    X[canonical_genes]
    .assign(sample_id=X.index, tp53_status=analysis_df["tp53_status"].values, tp53_mutant=analysis_df["tp53_mutant"].values)
    .melt(id_vars=["sample_id", "tp53_status", "tp53_mutant"], var_name="gene", value_name="expression")
)
canonical_long["known_role"] = canonical_long["gene"].map(CANONICAL_P53_TARGETS_INFO)
canonical_long.to_csv(TABLE_DIR / "canonical_p53_target_expression_long.csv", index=False)

n_cols = 4
n_rows = int(np.ceil(len(canonical_genes) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3.2 * n_rows), squeeze=False)
for ax, gene in zip(axes.ravel(), canonical_genes):
    data = canonical_long.loc[canonical_long["gene"] == gene]
    sns.boxplot(data=data, x="tp53_status", y="expression", hue="tp53_status", legend=False, ax=ax)
    ax.set_title(gene)
    ax.set_xlabel("")
    ax.set_ylabel("log2(TPM + 1)")
for ax in axes.ravel()[len(canonical_genes):]:
    ax.axis("off")
fig.suptitle("Canonical p53 target expression by TP53 status", y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / "canonical_p53_targets_by_status.png", dpi=180)
plt.show()

In [ ]:
def benjamini_hochberg(p_values):
    p_values = np.asarray(p_values, dtype=float)
    n = len(p_values)
    order = np.argsort(p_values)
    ranks = np.arange(1, n + 1)
    adjusted = np.empty(n, dtype=float)
    adjusted[order] = p_values[order] * n / ranks
    adjusted_sorted = adjusted[order]
    adjusted_sorted = np.minimum.accumulate(adjusted_sorted[::-1])[::-1]
    adjusted[order] = np.clip(adjusted_sorted, 0, 1)
    return adjusted

rows = []
for gene in canonical_genes:
    mut_values = X.loc[analysis_df["tp53_mutant"] == 1, gene]
    wt_values = X.loc[analysis_df["tp53_mutant"] == 0, gene]
    u_stat, p_val = mannwhitneyu(mut_values, wt_values, alternative="two-sided")
    rows.append({
        "gene": gene,
        "known_role": CANONICAL_P53_TARGETS_INFO[gene],
        "mean_mutant": mut_values.mean(),
        "mean_wt": wt_values.mean(),
        "median_mutant": mut_values.median(),
        "median_wt": wt_values.median(),
        "difference_mutant_minus_wt": mut_values.mean() - wt_values.mean(),
        "mannwhitney_u": u_stat,
        "p_value": p_val,
    })

canonical_tests = pd.DataFrame(rows)
canonical_tests["fdr_bh"] = benjamini_hochberg(canonical_tests["p_value"])
canonical_tests["direction"] = np.where(canonical_tests["difference_mutant_minus_wt"] < 0, "lower in mutant", "higher in mutant")
canonical_tests = canonical_tests.sort_values("fdr_bh")
canonical_tests.to_csv(TABLE_DIR / "canonical_p53_target_tests.csv", index=False)
display(canonical_tests)

fig, ax = plt.subplots(figsize=(8, max(4, 0.35 * len(canonical_tests))))
plot_df = canonical_tests.sort_values("difference_mutant_minus_wt")
colors = np.where(plot_df["difference_mutant_minus_wt"] < 0, "#4C78A8", "#E45756")
ax.barh(plot_df["gene"], plot_df["difference_mutant_minus_wt"], color=colors)
ax.axvline(0, color="black", linewidth=1)
ax.set_title("Canonical p53 target mean expression difference")
ax.set_xlabel("Mutant mean minus WT mean")
fig.tight_layout()
fig.savefig(FIG_DIR / "canonical_p53_target_directionality.png", dpi=180)
plt.show()

## 9. Sample-Sample Correlation Heatmaps

Correlation heatmaps give a compact view of whether samples are globally similar within TP53 groups. To keep the plot readable, we use p53 pathway genes when enough are present; otherwise we use top variable genes.

In [ ]:
corr_genes = [gene for gene in P53_PATHWAY_GENES if gene in X.columns]
if len(corr_genes) < 5:
    corr_genes = selected_genes[:200]
else:
    corr_genes = corr_genes[:200]

rng = np.random.RandomState(RANDOM_STATE)
max_samples_per_group = 60
sample_ids = []
for status_value in [0, 1]:
    ids = analysis_df.index[analysis_df["tp53_mutant"] == status_value].to_numpy()
    if len(ids) > max_samples_per_group:
        ids = rng.choice(ids, size=max_samples_per_group, replace=False)
    sample_ids.extend(ids.tolist())

corr_input = X.loc[sample_ids, corr_genes]
corr_matrix = corr_input.T.corr()
status_colors = analysis_df.loc[sample_ids, "tp53_status"].map({"WT": "#4C78A8", "Mutant": "#E45756"})

plt.figure(figsize=(9, 8))
sns.heatmap(corr_matrix, cmap="viridis", xticklabels=False, yticklabels=False, cbar_kws={"label": "Pearson correlation"})
plt.title(f"Sample-sample correlation using {len(corr_genes)} pathway/top-variable genes")
plt.tight_layout()
plt.savefig(FIG_DIR / "sample_sample_correlation_heatmap.png", dpi=180)
plt.show()

corr_summary = pd.DataFrame({
    "sample_id": sample_ids,
    "tp53_status": analysis_df.loc[sample_ids, "tp53_status"].values,
})
corr_summary.to_csv(TABLE_DIR / "correlation_heatmap_samples.csv", index=False)
print(f"Correlation heatmap used {len(sample_ids)} samples and {len(corr_genes)} genes.")

## 10. Exploratory Clustering

KMeans is not a classifier here. It asks whether unsupervised expression structure lines up with TP53 status or mutation type. Low agreement is still informative: it means TP53 labels are not the dominant source of global expression variance.

In [ ]:
def best_cluster_mapping(y_true, clusters):
    labels_true = pd.Series(y_true).astype(str).to_numpy()
    true_classes = np.unique(labels_true)
    cluster_classes = np.unique(clusters)
    cm = np.zeros((len(true_classes), len(cluster_classes)), dtype=int)
    for i, true_class in enumerate(true_classes):
        for j, cluster_class in enumerate(cluster_classes):
            cm[i, j] = np.sum((labels_true == true_class) & (clusters == cluster_class))
    row_ind, col_ind = linear_sum_assignment(-cm)
    mapping = {cluster_classes[col]: true_classes[row] for row, col in zip(row_ind, col_ind)}
    mapped = np.array([mapping.get(c, str(c)) for c in clusters])
    return true_classes, mapped, cm

cluster_features = pcs[:, :min(20, n_pcs)]
cluster_rows = []

# Binary status clustering.
kmeans_binary = KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init=20)
clusters_binary = kmeans_binary.fit_predict(cluster_features)
true_classes, mapped_binary, cm_binary = best_cluster_mapping(analysis_df["tp53_status"], clusters_binary)
cluster_rows.append({
    "target": "tp53_status",
    "n_clusters": 2,
    "adjusted_rand_index": adjusted_rand_score(analysis_df["tp53_status"], clusters_binary),
    "silhouette_score": silhouette_score(cluster_features, clusters_binary),
})

# Mutation-type clustering, if there are not too many classes.
y_type = analysis_df["mutation_type_collapsed"].astype(str)
n_type_classes = y_type.nunique()
if 2 <= n_type_classes <= 10:
    kmeans_type = KMeans(n_clusters=n_type_classes, random_state=RANDOM_STATE, n_init=20)
    clusters_type = kmeans_type.fit_predict(cluster_features)
    true_type_classes, mapped_type, cm_type = best_cluster_mapping(y_type, clusters_type)
    cluster_rows.append({
        "target": "mutation_type_collapsed",
        "n_clusters": n_type_classes,
        "adjusted_rand_index": adjusted_rand_score(y_type, clusters_type),
        "silhouette_score": silhouette_score(cluster_features, clusters_type),
    })
else:
    true_type_classes, cm_type = None, None

cluster_metrics = pd.DataFrame(cluster_rows)
cluster_metrics.to_csv(TABLE_DIR / "exploratory_clustering_metrics.csv", index=False)
display(cluster_metrics)

fig, ax = plt.subplots(figsize=(4, 3.5))
sns.heatmap(cm_binary, annot=True, fmt="d", cmap="Blues", xticklabels=np.unique(clusters_binary), yticklabels=true_classes, ax=ax)
ax.set_title("KMeans clusters vs TP53 status")
ax.set_xlabel("Cluster")
ax.set_ylabel("TP53 status")
fig.tight_layout()
fig.savefig(FIG_DIR / "kmeans_vs_tp53_status.png", dpi=180)
plt.show()

if cm_type is not None:
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm_type, annot=True, fmt="d", cmap="Blues", xticklabels=np.unique(clusters_type), yticklabels=true_type_classes, ax=ax)
    ax.set_title("KMeans clusters vs mutation type")
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Mutation type")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "kmeans_vs_mutation_type.png", dpi=180)
    plt.show()

## 11. Review Checklist

After running the notebook, decide which results are worth moving into the final report:

- Keep sparsity if mutant and WT groups differ meaningfully or if it exposes data quality issues.
- Keep outlier detection if flagged samples are interpretable and do not simply remove them without a sensitivity check.
- Keep lineage plots because they directly support the confounding discussion.
- Keep canonical p53 target expression because it connects EDA to biological interpretation.
- Treat clustering as optional. It is useful if you want to show that TP53 status is not the dominant unsupervised axis.